In [1]:
# 06_RESULT — PART1 / CELL 01
# CROSS-PHASE INTEGRITY & RESULTS BLUEPRINT
#
# SOURCE READ-ONLY | NO TRAINING | NO OVERWRITE
# 348-ID RECONCILIATION | 17 FIGURE HASH CHECK
# IMMUTABLE RESULTS + SHA256 AUDIT
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import uuid

import numpy as np
import pandas as pd

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")

MOD = ROOT / "04_Modeling" / "outputs"
TMP = ROOT / "05_Temporal" / "outputs"
OUT = ROOT / "06_Result" / "outputs" / "cross_phase_integrity"

MODEL = MOD / "leakage_grouping" / "modeling_labeled_348_grouped.csv"
TEMPORAL = TMP / "timestamp_canonicalization" / "temporal_labeled_348_locked.csv"
OOF = MOD / "comparative_evaluation" / "paired_oof_comparison_348.csv"
METRICS = MOD / "comparative_evaluation" / "model_comparison_metrics.csv"
DAILY = TMP / "daily_aggregation" / "daily_sentiment_counts_20_days.csv"

MODEL_FINDINGS = MOD / "final_evaluation_report" / "final_research_findings.json"
TEMP_FINDINGS = TMP / "comparative_temporal_report" / "temporal_research_findings.json"
MODEL_AUDIT = MOD / "final_evaluation_report" / "part20_final_report_audit.json"
TEMP_AUDIT = TMP / "comparative_temporal_report" / "part5_comparative_audit.json"

MODEL_REG = MOD / "final_evaluation_report" / "final_figure_registry.csv"
TEMP_REG = TMP / "timeline_visualizations" / "temporal_figure_registry.csv"

MODEL_FIG_DIR = MOD / "lstm_visualizations"
TEMP_FIG_DIR = TMP / "timeline_visualizations"

RECON_OUT = OUT / "cross_phase_dataset_reconciliation_348.csv"
METRICS_OUT = OUT / "locked_model_metrics.csv"
FIGURES_OUT = OUT / "integrated_figure_registry_17.csv"
BLUEPRINT_OUT = OUT / "results_chapter_blueprint.md"
AUDIT_OUT = OUT / "part1_cross_phase_integrity_audit.json"

OUTPUTS = [
    RECON_OUT,
    METRICS_OUT,
    FIGURES_OUT,
    BLUEPRINT_OUT,
    AUDIT_OUT
]

EXPECTED = {
    MODEL: "cfa7df27079ef47bdb1b0cb526bc017aa1c8fe18dc4d91fef870a3441e2e9fb7",
    TEMPORAL: "4ade3c2d0832f929e0a70ec0ae921653abdcb2aecf715a76fe0c274e674b5267",
    OOF: "670a444614a124c1b80106aa9c733c0d6b1cb690fd0ab0f20c9841d431323b1a",
    METRICS: "712aa656e0e5cee36db6f11ac4c94b4e1295cb93f87faac71e57f521091e3c57",
    DAILY: "4e4e1fd9e8b4e25b1aef729905b81306dd55a0d8053a2b509cee5d8acf89df21",
}

CASE_COUNTS = {
    "lombok_2018": 101,
    "palu_2018": 105,
    "sumatra_2025": 142
}
LABEL_COUNTS = {
    "NETRAL": 225,
    "POSITIF": 108,
    "NEGATIF": 15
}

def file_sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def relative(path):
    return path.relative_to(ROOT).as_posix()

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def write_json(path, obj):
    path.write_text(
        json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False),
        encoding="utf-8"
    )

def load_csv(path):
    return pd.read_csv(
        path, encoding="utf-8-sig",
        dtype={"id": "string"}, keep_default_na=False
    )

print("=" * 75)
print("06_RESULT PART1 — CROSS-PHASE PREFLIGHT")
print("=" * 75)

# ------------------------------------------------------------
# A. VERIFY LOCKED SOURCES
# ------------------------------------------------------------

SOURCES = list(EXPECTED) + [
    MODEL_FINDINGS, TEMP_FINDINGS,
    MODEL_AUDIT, TEMP_AUDIT,
    MODEL_REG, TEMP_REG
]

for path in SOURCES:
    assert path.is_file(), f"Missing source: {path}"

for path in OUTPUTS:
    if path.exists():
        raise FileExistsError(
            f"Existing output; refusing overwrite: {path}"
        )

source_before = {
    relative(p): file_sha(p)
    for p in SOURCES
}

for path, expected_sha in EXPECTED.items():
    assert file_sha(path) == expected_sha, (
        f"Locked SHA256 mismatch: {relative(path)}"
    )

model_audit = read_json(MODEL_AUDIT)
temporal_audit = read_json(TEMP_AUDIT)
model_findings = read_json(MODEL_FINDINGS)
temporal_findings = read_json(TEMP_FINDINGS)

assert model_audit["status"] == "PART20_FINAL_REPORT_AUDIT_PASS"
assert temporal_audit["status"] == "TEMPORAL_PART5_COMPARATIVE_AUDIT_PASS"
assert model_findings["dataset_rows"] == 348
assert temporal_findings["sample_size"] == 348
assert temporal_findings["comparison_type"] == "descriptive"

print("LOCKED SOURCE HASHES AND FINAL AUDITS: PASS")

# ------------------------------------------------------------
# B. RECONCILE ALL 348 IDs, CASES, LABELS, GROUPS
# ------------------------------------------------------------

model = load_csv(MODEL)
temporal = load_csv(TEMPORAL)
oof = load_csv(OOF)

assert len(model) == len(temporal) == len(oof) == 348

for df, name in [
    (model, "model"),
    (temporal, "temporal"),
    (oof, "oof")
]:
    assert df["id"].notna().all(), f"Missing ID: {name}"
    assert df["id"].is_unique, f"Duplicate IDs: {name}"

assert model["case"].value_counts().to_dict() == CASE_COUNTS
assert temporal["case"].value_counts().to_dict() == CASE_COUNTS
assert model["sentiment_label"].value_counts().to_dict() == LABEL_COUNTS
assert temporal["sentiment_label"].value_counts().to_dict() == LABEL_COUNTS

base = model[[
    "id", "case", "sentiment_label", "leakage_group_id"
]].copy()

base = base.rename(columns={
    "case": "case_model",
    "sentiment_label": "label_model",
    "leakage_group_id": "group_model"
})

temp_sub = temporal[[
    "id", "case", "sentiment_label",
    "leakage_group_id", "date_wib"
]].rename(columns={
    "case": "case_temporal",
    "sentiment_label": "label_temporal",
    "leakage_group_id": "group_temporal"
})

oof_sub = oof[[
    "id", "case", "true_label",
    "leakage_group_id", "validation_fold",
    "predicted_label_baseline", "predicted_label_lstm"
]].rename(columns={
    "case": "case_oof",
    "true_label": "label_oof",
    "leakage_group_id": "group_oof"
})

recon = base.merge(
    temp_sub, on="id", how="outer",
    indicator="merge_model_temporal",
    validate="one_to_one"
)

recon = recon.merge(
    oof_sub, on="id", how="outer",
    indicator="merge_with_oof",
    validate="one_to_one"
)

assert len(recon) == 348
assert recon["merge_model_temporal"].eq("both").all()
assert recon["merge_with_oof"].eq("both").all()

checks = {
    "case_match": ("case_model", "case_temporal", "case_oof"),
    "label_match": ("label_model", "label_temporal", "label_oof"),
    "group_match": ("group_model", "group_temporal", "group_oof")
}

for check_name, columns in checks.items():
    recon[check_name] = (
        recon[columns[0]].astype(str).eq(
            recon[columns[1]].astype(str)
        )
        &
        recon[columns[0]].astype(str).eq(
            recon[columns[2]].astype(str)
        )
    )
    assert recon[check_name].all(), (
        f"Cross-phase mismatch: {check_name}"
    )

assert recon["date_wib"].astype(str).str.fullmatch(
    r"\d{4}-\d{2}-\d{2}"
).all()

assert recon["validation_fold"].nunique() == 3

assert recon["predicted_label_baseline"].isin(
    LABEL_COUNTS
).all()

assert recon["predicted_label_lstm"].isin(
    LABEL_COUNTS
).all()

recon = recon.sort_values(
    ["case_model", "id"],
    kind="mergesort"
).reset_index(drop=True)

print("348-ID MODELING/TEMPORAL/OOF JOIN: PASS")
print("CASE, LABEL, LEAKAGE GROUP RECONCILIATION: PASS")

# ------------------------------------------------------------
# C. VERIFY MODEL METRICS
# ------------------------------------------------------------

metrics = pd.read_csv(METRICS, encoding="utf-8-sig")

assert len(metrics) == 2
assert set(metrics["model"]) == {"baseline", "lstm"}
assert (metrics["rows"] == 348).all()

by_model = metrics.set_index("model")

for name in ["baseline", "lstm"]:
    for key in ["accuracy", "macro_f1", "macro_recall"]:
        assert np.isclose(
            float(by_model.loc[name, key]),
            float(model_findings[name][key]),
            rtol=0, atol=1e-12
        ), f"Metric inconsistency: {name}/{key}"

# Independently reconstruct accuracy from OOF predictions.
for name, pred_col in [
    ("baseline", "predicted_label_baseline"),
    ("lstm", "predicted_label_lstm")
]:
    accuracy = (
        recon[pred_col].astype(str)
        == recon["label_model"].astype(str)
    ).mean()

    assert np.isclose(
        float(accuracy),
        float(by_model.loc[name, "accuracy"]),
        rtol=0, atol=1e-12
    )

assert (
    float(by_model.loc["baseline", "macro_f1"])
    > float(by_model.loc["lstm", "macro_f1"])
)

print("BASELINE/LSTM METRICS AND OOF ACCURACY: PASS")

# ------------------------------------------------------------
# D. RECONCILE TEMPORAL AGGREGATION
# ------------------------------------------------------------

daily = pd.read_csv(DAILY, encoding="utf-8-sig")

assert len(daily) == 20
assert int(daily["total_tweets"].sum()) == 348

for case, expected in CASE_COUNTS.items():
    part = daily.loc[daily["case"] == case]
    assert int(part["total_tweets"].sum()) == expected

    matching = recon.loc[recon["case_model"] == case]
    assert len(matching) == expected

    for label in LABEL_COUNTS:
        actual = int(part[f"count_{label}"].sum())
        expected_count = int(
            (matching["label_model"] == label).sum()
        )
        assert actual == expected_count, (
            f"Temporal mismatch: {case}/{label}"
        )

# Exact day/label match, not just total counts.
daily_from_tweets = (
    recon.groupby(
        ["case_model", "date_wib", "label_model"]
    ).size()
)

for row in daily.itertuples(index=False):
    for label in LABEL_COUNTS:
        expected_count = int(daily_from_tweets.get(
            (row.case, row.date_wib, label), 0
        ))
        assert int(getattr(row, f"count_{label}")) == expected_count

print("DAILY SENTIMENT RECONCILIATION: PASS")

# ------------------------------------------------------------
# E. VERIFY AND INTEGRATE 17 EXISTING FIGURES
# ------------------------------------------------------------

model_registry = pd.read_csv(
    MODEL_REG, encoding="utf-8-sig"
)
temp_registry = pd.read_csv(
    TEMP_REG, encoding="utf-8-sig"
)

assert len(model_registry) == 9
assert len(temp_registry) == 8

figure_rows = []
figure_sources = []

for row in model_registry.itertuples(index=False):
    path = MODEL_FIG_DIR / row.filename

    assert path.is_file(), f"Missing figure: {path}"
    assert file_sha(path) == row.sha256
    assert path.open("rb").read(8) == bytes.fromhex(
        "89504e470d0a1a0a"
    )

    figure_sources.append(path)
    figure_rows.append({
        "integrated_figure_id": f"MODEL_{row.figure_id}",
        "stage": "04_Modeling",
        "filename": row.filename,
        "relative_repository_path": relative(path),
        "sha256": file_sha(path),
        "usage": "modeling_results"
    })

for row in temp_registry.itertuples(index=False):
    path = TEMP_FIG_DIR / row.filename

    assert path.is_file(), f"Missing figure: {path}"
    assert file_sha(path) == row.figure_sha256
    assert path.open("rb").read(8) == bytes.fromhex(
        "89504e470d0a1a0a"
    )

    figure_sources.append(path)
    figure_rows.append({
        "integrated_figure_id": f"TEMP_{row.figure_id}",
        "stage": "05_Temporal",
        "filename": row.filename,
        "relative_repository_path": relative(path),
        "sha256": file_sha(path),
        "usage": "temporal_results"
    })

figure_registry = pd.DataFrame(figure_rows)

assert len(figure_registry) == 17
assert figure_registry["integrated_figure_id"].is_unique

figure_before = {
    relative(p): file_sha(p) for p in figure_sources
}

print("9 MODELING + 8 TEMPORAL FIGURE HASHES: PASS")

# ------------------------------------------------------------
# F. RESULTS CHAPTER BLUEPRINT
# ------------------------------------------------------------

blueprint = """# BLUEPRINT BAB IV — HASIL DAN PEMBAHASAN

**Penelitian:** Sentimen Penetapan Bencana Nasional

**Status:** Blueprint awal berdasarkan hasil audit lintas fase.

> Struktur ini dapat disesuaikan dengan proposal dan
> pedoman penulisan kampus. Belum merupakan naskah
> BAB IV final.

## 4.1 Gambaran Umum Dataset

- Jelaskan sumber dan proses seleksi data.
- Sajikan 348 tweet berlabel manusia.
- Distribusi kasus: Lombok 101, Palu 105, Sumatera 142.
- Distribusi sentimen: NETRAL 225, POSITIF 108, NEGATIF 15.
- Jelaskan 337 leakage groups dan pengendalian leakage.
- Jelaskan keterbatasan cakupan sampel.

## 4.2 Metode dan Pengaturan Evaluasi

- Stratified Group K-Fold tiga fold, seed 123.
- Baseline TF-IDF + Logistic Regression.
- Model LSTM_SMALL_V1.
- Metrik utama Out-of-Fold Macro-F1.
- Bedakan training, validation, dan OOF.
- Tidak ada independent holdout test set.

## 4.3 Hasil Pemodelan Sentimen

- Sajikan metrik kedua model.
- Baseline: accuracy 0.7040, Macro-F1 0.5236.
- LSTM: accuracy 0.5862, Macro-F1 0.4751.
- Bahas grafik hasil modeling dan confusion matrix.
- Jelaskan indikasi overfitting LSTM.
- Bahas kelemahan kedua model pada kelas NEGATIF.

## 4.4 Analisis Kesalahan Klasifikasi

- Kedua model benar: 165 tweet.
- Hanya baseline benar: 80 tweet.
- Hanya LSTM benar: 39 tweet.
- Kedua model salah: 64 tweet.
- Jelaskan perbedaan precision dan recall NEGATIF.
- Jangan menyatakan signifikansi statistik tanpa pengujian.

## 4.5 Distribusi Sentimen dan Analisis Temporal

- Gunakan label manusia, bukan prediksi model.
- Sajikan jumlah dan persentase harian per kasus.
- Tampilkan 20 hari kalender kasus.
- Sebanyak 15 hari memiliki observasi dan 5 hari kosong.
- Gunakan grafik temporal asli Part4.
- Hari relatif 0 berarti awal observasi, bukan awal bencana.

## 4.6 Pembahasan Perbandingan Kasus

- Bandingkan Lombok, Palu, dan Sumatera secara deskriptif.
- Bedakan jumlah tweet dan proporsi sentimen.
- Jelaskan variasi cakupan waktu dan distribusi sampel.
- Hindari generalisasi sebagai opini seluruh masyarakat.
- Hindari klaim hubungan sebab-akibat kebijakan.

## 4.7 Keterbatasan Penelitian

- Hanya 348 tweet berlabel.
- Kelas NEGATIF sangat kecil (15 tweet).
- Perbedaan jendela pengamatan antarkasus.
- Pengaruh strategi scraping dan seleksi sampel.
- Tidak tersedia independent holdout.
- Potensi overfitting pada LSTM.
- Hasil temporal bersifat deskriptif.

## 4.8 Ringkasan Temuan

- Baseline lebih tinggi menurut OOF Macro-F1.
- LSTM memiliki Macro-Recall lebih tinggi.
- NETRAL mendominasi sampel ketiga kasus.
- Pola temporal berbeda menurut kasus.
- Simpulan terbatas pada sampel dan metode penelitian.

## Catatan Integrasi

Modeling menjelaskan performa prediksi.
Temporal menjelaskan distribusi label manusia.
Keduanya menggunakan 348 tweet yang sama,
tetapi menjawab pertanyaan analitis berbeda.

Dokumen ini merupakan blueprint.
Finalisasi BAB IV harus disesuaikan dengan
rumusan masalah, tujuan, serta pedoman akademik.
"""

# ------------------------------------------------------------
# G. AUDIT MANIFEST
# ------------------------------------------------------------

now = datetime.now(timezone.utc).isoformat()

audit = {
    "stage": "06_Result",
    "part": "1 Cross-Phase Integrity & Blueprint",
    "generated_at_utc": now,
    "modeling_source_commit":
        "5cf80cc802f99c25fc4b714a4036beea3ae930e4",
    "temporal_source_commit":
        "7571fc02d68cdb2f81b404554df4304940e2bb07",
    "dataset_rows": 348,
    "unique_ids": 348,
    "case_match_count": int(recon["case_match"].sum()),
    "label_match_count": int(recon["label_match"].sum()),
    "leakage_group_match_count":
        int(recon["group_match"].sum()),
    "oof_rows": 348,
    "daily_rows": 20,
    "daily_tweets": 348,
    "model_figures_verified": 9,
    "temporal_figures_verified": 8,
    "total_figures_verified": 17,
    "source_sha256": source_before,
    "figure_sha256": figure_before,
    "model_training_performed": False,
    "source_modified": False,
    "results_blueprint_prepared": True,
    "output_sha256": {},
    "status": "RESULT_PART1_CROSS_PHASE_INTEGRITY_PASS"
}

# ------------------------------------------------------------
# H. IMMUTABLE EXPORT
# ------------------------------------------------------------

OUT.mkdir(parents=True, exist_ok=True)

for path in OUTPUTS:
    if path.exists():
        raise FileExistsError(
            f"Refusing to overwrite existing output: {path}"
        )

token = uuid.uuid4().hex[:10]
temps = {
    p: OUT / f".result_p1_{token}_{i}.tmp"
    for i, p in enumerate(OUTPUTS)
}
created = []

try:
    recon.to_csv(
        temps[RECON_OUT],
        index=False, encoding="utf-8-sig"
    )

    metrics.to_csv(
        temps[METRICS_OUT],
        index=False, encoding="utf-8-sig"
    )

    figure_registry.to_csv(
        temps[FIGURES_OUT],
        index=False, encoding="utf-8-sig"
    )

    temps[BLUEPRINT_OUT].write_text(
        blueprint, encoding="utf-8"
    )

    audit["output_sha256"] = {
        p.name: file_sha(temps[p])
        for p in OUTPUTS[:-1]
    }

    write_json(temps[AUDIT_OUT], audit)

    # Check no source changed before publication.
    assert {
        relative(p): file_sha(p)
        for p in SOURCES
    } == source_before

    assert {
        relative(p): file_sha(p)
        for p in figure_sources
    } == figure_before

    # Re-open saved outputs and verify row counts.
    assert len(pd.read_csv(temps[RECON_OUT])) == 348
    assert len(pd.read_csv(temps[FIGURES_OUT])) == 17

    for destination in OUTPUTS:
        with destination.open("xb") as f:
            f.write(temps[destination].read_bytes())
        created.append(destination)

except Exception:
    # Remove only outputs created by this run.
    for path in created:
        if path.exists():
            path.unlink()
    raise

finally:
    for path in temps.values():
        if path.exists():
            path.unlink()

# ------------------------------------------------------------
# I. FINAL VERIFICATION
# ------------------------------------------------------------

for name, expected_sha in audit["output_sha256"].items():
    assert file_sha(OUT / name) == expected_sha

assert {
    relative(p): file_sha(p)
    for p in SOURCES
} == source_before

assert {
    relative(p): file_sha(p)
    for p in figure_sources
} == figure_before

assert read_json(AUDIT_OUT)["status"] == (
    "RESULT_PART1_CROSS_PHASE_INTEGRITY_PASS"
)

print("\n" + "=" * 75)
print("06_RESULT PART1 — FINAL AUDIT")
print("=" * 75)

print("Cross-phase rows     :", len(recon))
print("Matching IDs         :", recon["id"].nunique())
print("Matching cases       :", int(recon["case_match"].sum()))
print("Matching labels      :", int(recon["label_match"].sum()))
print("Matching groups      :", int(recon["group_match"].sum()))
print("Modeling figures     :", 9)
print("Temporal figures     :", 8)
print("Total figures        :", len(figure_registry))
print("Source modified      : NO")
print("Training performed   : NO")

print("\nLOCKED MODEL METRICS")
display(metrics)

print("\nINTEGRITY CHECK COUNTS")
display(
    recon[[
        "case_match", "label_match", "group_match"
    ]].sum().to_frame("matching_rows")
)

print("\nARTIFACTS:")
for p in OUTPUTS:
    print("-", p)

print("\nFINAL STATUS:", audit["status"])
print("=" * 75)


06_RESULT PART1 — CROSS-PHASE PREFLIGHT
LOCKED SOURCE HASHES AND FINAL AUDITS: PASS
348-ID MODELING/TEMPORAL/OOF JOIN: PASS
CASE, LABEL, LEAKAGE GROUP RECONCILIATION: PASS
BASELINE/LSTM METRICS AND OOF ACCURACY: PASS
DAILY SENTIMENT RECONCILIATION: PASS
9 MODELING + 8 TEMPORAL FIGURE HASHES: PASS

06_RESULT PART1 — FINAL AUDIT
Cross-phase rows     : 348
Matching IDs         : 348
Matching cases       : 348
Matching labels      : 348
Matching groups      : 348
Modeling figures     : 9
Temporal figures     : 8
Total figures        : 17
Source modified      : NO
Training performed   : NO

LOCKED MODEL METRICS


,model,rows,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1
0,baseline,348,0.704023,0.776108,0.494321,0.523555,0.689845
1,lstm,348,0.586207,0.475956,0.559630,0.475130,0.617452



INTEGRITY CHECK COUNTS


,matching_rows
case_match,348
label_match,348
group_match,348



ARTIFACTS:
- C:\Penelitian Sentimen Bencana Nasional\06_Result\outputs\cross_phase_integrity\cross_phase_dataset_reconciliation_348.csv
- C:\Penelitian Sentimen Bencana Nasional\06_Result\outputs\cross_phase_integrity\locked_model_metrics.csv
- C:\Penelitian Sentimen Bencana Nasional\06_Result\outputs\cross_phase_integrity\integrated_figure_registry_17.csv
- C:\Penelitian Sentimen Bencana Nasional\06_Result\outputs\cross_phase_integrity\results_chapter_blueprint.md
- C:\Penelitian Sentimen Bencana Nasional\06_Result\outputs\cross_phase_integrity\part1_cross_phase_integrity_audit.json

FINAL STATUS: RESULT_PART1_CROSS_PHASE_INTEGRITY_PASS
